# Low Chin Tone Detection — **Version 5 (Merged HIGH & LOW in XML)**
요청사항 반영:
- **연속된 HIGH/LOW**를 각각 **하나의 이벤트로 병합**하여 XML에 출력
- 슬라이딩 다중-스케일 RMS(0.2s/1.0s), 이동 median+MAD 임계치, 히스테리시스는 V4와 동일
- 이벤트 타임라인을 **프레임 단위 상태(High/Low)** 로 만든 뒤 **연속 구간을 합침(run-length merge)**
- XML에는 `HIGH_CHIN_TONE_x.xxx`, `LOW_CHIN_TONE_x.xxx` 모두 기록

In [ ]:

import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from datetime import timedelta, datetime
import xml.etree.ElementTree as ET

# sleep_stage EDF loader (same path as your environment)
try:
    sys.path.append('/home/honeynaps/data/shared/integrate/sleep_stage')
    from sleep_stage.modules.iofiles import edf as edf_io
    HAVE_SLEEP_STAGE = True
except Exception:
    HAVE_SLEEP_STAGE = False
    print("[WARN] sleep_stage not available here. Run this in your environment.")

In [ ]:

def butter_highpass(cut_hz, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cut_hz/nyq, btype='highpass')
    return b, a

def highpass_filter(x, fs, cut_hz=5.0, order=4):
    if not cut_hz or cut_hz <= 0: return x
    b, a = butter_highpass(cut_hz, fs, order)
    return filtfilt(b, a, x)

def rms_envelope(sig, fs, win_s=0.2, hop_s=0.1):
    win = max(1, int(round(win_s*fs)))
    hop = max(1, int(round(hop_s*fs)))
    sq = sig.astype(float)**2
    ker = np.ones(win)/float(win)
    mov = np.convolve(sq, ker, mode='same')
    env = np.sqrt(np.maximum(mov, 0.0))
    t = np.arange(len(env))/fs
    return t[::hop], env[::hop]

def rolling_median_mad(x, win):
    s = pd.Series(x)
    med = s.rolling(win, center=True, min_periods=1).median()
    mad = (s - med).abs().rolling(win, center=True, min_periods=1).median()
    mad_scaled = 1.4826 * mad
    med = med.fillna(method='bfill').fillna(method='ffill')
    mad_scaled = mad_scaled.fillna(method='bfill').fillna(method='ffill')
    if (mad_scaled==0).any():
        repl = float(np.median(mad_scaled[mad_scaled>0])) if (mad_scaled>0).any() else 1e-8
        mad_scaled = mad_scaled.replace(0, repl)
    return med.values, mad_scaled.values

In [ ]:

from dataclasses import dataclass

@dataclass
class DetectConfig:
    fs: int = 50
    hp_cut_hz: float = 5.0
    short_win_s: float = 0.2
    long_win_s: float = 1.0
    hop_s: float = 0.1
    baseline_win_s: float = 60.0
    z_start_short: float = 4.0
    z_end_short: float = 2.5
    z_start_long: float = 3.5
    z_end_long: float = 2.0
    r_start_short: float = 2.5
    r_end_short: float = 1.6
    r_start_long: float = 2.0
    r_end_long: float = 1.4
    min_event_dur_s: float = 0.10
    merge_gap_s: float = 0.15

def detect_emg_events(raw_emg, cfg: DetectConfig):
    fs = cfg.fs
    x = raw_emg.astype(float)
    x = x - np.median(x)
    if cfg.hp_cut_hz and cfg.hp_cut_hz>0:
        x = highpass_filter(x, fs, cut_hz=cfg.hp_cut_hz, order=4)
    # envelopes
    t_s, env_s = rms_envelope(x, fs, win_s=cfg.short_win_s, hop_s=cfg.hop_s)
    t_l, env_l = rms_envelope(x, fs, win_s=cfg.long_win_s,  hop_s=cfg.hop_s)
    # baselines (frames)
    base_win = max(3, int(round(cfg.baseline_win_s/cfg.hop_s)))
    b_s, mad_s = rolling_median_mad(env_s, base_win)
    b_l, mad_l = rolling_median_mad(env_l, base_win)
    eps=1e-8
    z_s = (env_s - b_s)/(mad_s+eps)
    z_l = (env_l - b_l)/(mad_l+eps)
    r_s = env_s/np.maximum(b_s, eps)
    r_l = env_l/np.maximum(b_l, eps)
    # hysteresis
    start_mask = (z_s>cfg.z_start_short) | (z_l>cfg.z_start_long) | (r_s>cfg.r_start_short) | (r_l>cfg.r_start_long)
    end_mask   = (z_s<cfg.z_end_short)  & (z_l<cfg.z_end_long)  & (r_s<cfg.r_end_short)  & (r_l<cfg.r_end_long)

    # frame-level state (True=HIGH)
    state = np.zeros_like(env_s, dtype=bool)
    in_evt=False
    for i in range(len(env_s)):
        if not in_evt and start_mask[i]:
            in_evt=True
        elif in_evt and end_mask[i]:
            in_evt=False
        state[i]=in_evt

    # make high events from state + merge small gaps
    events=[]
    hop=cfg.hop_s
    i=0
    while i<len(state):
        if state[i]:
            j=i+1
            while j<len(state) and state[j]: j+=1
            s=i*hop; e=j*hop
            if e-s>=cfg.min_event_dur_s: events.append((s,e))
            i=j
        else:
            i+=1
    # merge nearby highs
    merged=[]
    if events:
        cs,ce=events[0]
        for s,e in events[1:]:
            if s-ce<=cfg.merge_gap_s:
                ce=max(ce,e)
            else:
                merged.append((cs,ce)); cs,ce=s,e
        merged.append((cs,ce))

    # rebuild state from merged highs (ensures contiguous highs are one block)
    state2 = np.zeros_like(state, dtype=bool)
    for s,e in merged:
        i0=int(round(s/hop)); i1=int(round(e/hop))
        state2[i0:i1]=True

    debug = {
        "t_short": t_s, "env_short": env_s, "base_short": b_s,
        "t_long": t_l,  "env_long":  env_l, "base_long":  b_l,
        "state_high": state2
    }
    return debug

In [ ]:

def load_edf_raw(edf_path, fs=50):
    if not HAVE_SLEEP_STAGE:
        raise RuntimeError("sleep_stage not available. Run in your environment.")
    edf, _ = edf_io.load(
        path=edf_path, preload=True, resample=fs, preset="STAGENET",
        exclude=True, missing_ch='raise'
    )
    meas_date = edf.info['meas_date'].replace(tzinfo=None)
    data = edf.get_data()
    ch_names = edf.ch_names

    SID_MAP = { 'F3-':'F3_2','F4-':'F4_1','C3-':'C3_2','C4-':'C4_1',
                'O1-':'O1_2','O2-':'O2_1','LOC':'LOC','ROC':'ROC','EMG':'CHIN' }
    sigs = {}
    for i,nm in enumerate(ch_names):
        if nm in SID_MAP: sigs[SID_MAP[nm]]=data[i]
        else: sigs[SID_MAP.get(nm[:3], nm[:3])] = data[i]
    if 'CHIN' not in sigs: raise RuntimeError("CHIN channel not found")
    chin = sigs['CHIN'].astype(float) * 1000.0  # to µV
    return chin, meas_date, ch_names

In [ ]:

def runs_from_state(t_frames, env_short, state_high, total_len_s,
                    min_dur_s=0.5, merge_same_gap_s=0.0):
    """
    Build merged HIGH/LOW runs that cover the entire signal [0, total_len_s).
    - state_high: boolean per frame (True=HIGH)
    - merge_same_gap_s: if >0, merge adjacent same-type runs separated by ≤ gap (rare)
    Returns list of dicts with keys: start, end, duration, type, avg_rms.
    """
    hop = t_frames[1]-t_frames[0] if len(t_frames)>1 else 0.1
    N = len(state_high)
    # Convert to runs
    runs=[]
    if N==0:
        return runs
    cur_type = "HIGH" if state_high[0] else "LOW"
    cur_start = 0.0
    acc_vals = [env_short[0]]
    for i in range(1,N):
        typ = "HIGH" if state_high[i] else "LOW"
        if typ==cur_type:
            acc_vals.append(env_short[i])
        else:
            cur_end = i*hop
            dur = cur_end-cur_start
            if dur>=min_dur_s:
                runs.append({"start":cur_start,"end":cur_end,"duration":dur,
                             "type":cur_type,"avg_rms":float(np.mean(acc_vals))})
            # reset
            cur_type = typ
            cur_start = i*hop
            acc_vals = [env_short[i]]
    # last run
    cur_end = N*hop
    if cur_end>total_len_s: cur_end = total_len_s
    dur = cur_end-cur_start
    if dur>=min_dur_s:
        runs.append({"start":cur_start,"end":cur_end,"duration":dur,
                     "type":cur_type,"avg_rms":float(np.mean(acc_vals))})
    # Merge same-type runs separated by ≤ merge_same_gap_s
    if merge_same_gap_s>0 and runs:
        merged=[runs[0]]
        for r in runs[1:]:
            prev=merged[-1]
            if r['type']==prev['type'] and (r['start']-prev['end'])<=merge_same_gap_s:
                # merge
                total_dur = (prev['duration'] + (r['start']-prev['end']) + r['duration'])
                # weighted average RMS by duration (simple mean of means weighted by duration)
                w_prev = prev['duration']; w_r = r['duration']
                avg_rms = (prev['avg_rms']*w_prev + r['avg_rms']*w_r) / max(w_prev+w_r, 1e-8)
                prev['end']=r['end']; prev['duration']=prev['end']-prev['start']; prev['avg_rms']=avg_rms
            else:
                merged.append(r)
        runs=merged
    return runs

def save_merged_high_low_xml(meas_date, merged_runs, xml_path, total_len_s):
    root = ET.Element("annotationlist")
    ET.SubElement(root, "recording_duration").text = f"{total_len_s:.6f}"
    for ev in merged_runs:
        on = meas_date + timedelta(seconds=ev['start'])
        ann = ET.SubElement(root, "annotation")
        ET.SubElement(ann, "onset").text = on.strftime("%Y-%m-%dT%H:%M:%S.%f")
        ET.SubElement(ann, "duration").text = f"{ev['duration']:.6f}"
        ET.SubElement(ann, "description").text = f"{ev['type']}_CHIN_TONE_{ev['avg_rms']:.4f}"
        ET.SubElement(ann, "location").text = "EEG-EMG"
    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)
    with open(xml_path,"wb") as fp:
        fp.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
        tree.write(fp, encoding="UTF-8", xml_declaration=False)
    return len(merged_runs)

In [ ]:

# ====== CONFIG ======
fs = 50
cfg = DetectConfig(fs=fs)
EDF_DIR = '/home/honeynaps/data/GOLDEN/EDF2'
OUTPUT_DIR = './output_v5_mergedHL'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_FILES = 5  # set None to process all

edf_files = [f for f in os.listdir(EDF_DIR) if f.lower().endswith('.edf')]
edf_files.sort()
if MAX_FILES is not None: edf_files = edf_files[:MAX_FILES]
print(f"Found {len(edf_files)} EDF files")

summary=[]
for i,fname in enumerate(edf_files,1):
    print(f"\n[{i}/{len(edf_files)}] {fname}")
    chin, meas_date, ch_names = load_edf_raw(os.path.join(EDF_DIR, fname), fs=fs)
    total_len_s = len(chin)/fs
    dbg = detect_emg_events(chin, cfg)
    t_frames = dbg['t_short']; env_short = dbg['env_short']; state_high = dbg['state_high']

    # Build merged HIGH/LOW timeline
    runs = runs_from_state(t_frames, env_short, state_high, total_len_s,
                           min_dur_s=0.5, merge_same_gap_s=0.0)  # 0.5s minimum per run

    # Save XML with merged HIGH/LOW
    base = os.path.splitext(fname)[0]
    xml_path = os.path.join(OUTPUT_DIR, f"{base}_CHIN_MERGED_HL_V5.xml")
    n = save_merged_high_low_xml(meas_date, runs, xml_path, total_len_s)
    print(f"  Merged runs saved: {n} -> {xml_path}")

    # Report quick stats
    n_high = sum(1 for r in runs if r['type']=="HIGH")
    n_low  = sum(1 for r in runs if r['type']=="LOW")
    d_high = sum(r['duration'] for r in runs if r['type']=="HIGH")
    d_low  = sum(r['duration'] for r in runs if r['type']=="LOW")
    summary.append((fname, n_high, n_low, d_high, d_low, xml_path))

print("\n=== SUMMARY ===")
for row in summary:
    fname, nH, nL, dH, dL, path = row
    print(f"{fname}: HIGH n={nH}, dur={dH:.1f}s | LOW n={nL}, dur={dL:.1f}s -> {path}")
print("Done.")

In [ ]:

# Optional quick plot for QA on the last file processed (if arrays in scope)
try:
    start_s, dur_s = 30.0, 30.0
    t = np.arange(len(chin))/fs
    m0 = int(round(start_s*fs)); m1=int(round((start_s+dur_s)*fs))

    plt.figure(figsize=(12,5))
    plt.plot(t[m0:m1], chin[m0:m1], linewidth=0.8)
    plt.title("Raw CHIN EMG"); plt.xlabel("Time (s)"); plt.ylabel("µV")
    plt.show()

    mask = (t_frames>=start_s)&(t_frames<=start_s+dur_s)
    plt.figure(figsize=(12,5))
    plt.plot(t_frames[mask], env_short[mask], label="env_short")
    plt.plot(t_frames[mask], dbg['base_short'][mask], label="base_short", linestyle='--')
    # shade HIGH runs
    for r in runs:
        if r['type']!="HIGH": continue
        if r['end']<start_s or r['start']>start_s+dur_s: continue
        plt.axvspan(max(r['start'], start_s), min(r['end'], start_s+dur_s), alpha=0.2)
    plt.legend(); plt.title("Short RMS envelope with HIGH run shading")
    plt.xlabel("Time (s)"); plt.ylabel("RMS (µV)")
    plt.show()
except Exception as e:
    print("Visualization skipped:", e)